# LABD GRPO Training

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

# Override os.cpu_count to return actual allocated cores
import multiprocessing
multiprocessing.cpu_count = lambda: 4

In [3]:
%%capture
import os, glob, shutil

# Fix Modal missing curand headers
for h in glob.glob("/usr/local/lib/python3.12/site-packages/nvidia/curand/include/*.h"):
    dest = f"/usr/local/cuda/include/{os.path.basename(h)}"
    if not os.path.exists(dest):
        os.symlink(h, dest)

# Clear stale flashinfer build cache
for d in [os.path.expanduser("~/.cache/flashinfer"),
          "/root/unsloth_compiled_cache/.cache/flashinfer"]:
    if os.path.exists(d):
        shutil.rmtree(d)

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Then install packages
!pip install unsloth vllm -qqq
!pip install langid -qq
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install pydantic==2.11.5 pydantic_core==2.33.2 --force-reinstall --no-deps
!pip install "protobuf>=6.33.5" -qqq

In [2]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
torch.multiprocessing.set_start_method('spawn', force=True)
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset, Dataset
import io
import sys
import signal
import contextlib
import re
import json
import random
import ast
import traceback
import resource
import concurrent.futures
import gc
import subprocess
import tempfile

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Model Loading (with vLLM for GRPO generation)

In [3]:
from huggingface_hub import login
from getpass import getpass

# Option 1: Interactive (Best for shared notebooks)
hf_token = getpass("Enter your Hugging Face Token: ")
login(token=hf_token)

In [4]:
# Optimized for L4 24GB VRAM
max_seq_length = 6144
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    "moazeldegwy/Qwen3-8B-LABD",
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16,
    load_in_4bit = False,
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.85,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

INFO 03-08 17:32:37 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.17.0.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 47.374 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.87875.
Unsloth: vLLM loading moazeldegwy/Qwen3-8B-LABD with actual GPU utilization = 87.02%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 47.37 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 6144. Num Sequences = 96.
Unsloth: vLLM's KV Cache can use up to 25.74 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `use_cudagraph` is not sup

/usr/local/lib/python3.12/site-packages/pydantic/type_adapter.py:572: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-08 17:32:50 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-08 17:32:50 [model.py:1554] Using max model len 6144
INFO 03-08 17:32:50 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-08 17:32:50 [vllm.py:747] Asynchronous scheduling is enabled.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

INFO 03-08 17:36:16 [gpu_model_runner.py:5360] Graph capturing finished in 15 secs, took 0.71 GiB
INFO 03-08 17:36:16 [vllm_utils.py:736] Unsloth: Patched vLLM v1 graph capture finished in 15 secs.


INFO 03-08 17:36:17 [core.py:282] init engine (profile, create kv cache, warmup model) took 121.57 seconds
INFO 03-08 17:36:19 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'post_layernorm', 'norm', 'norm2', 'q_norm', 'layer_norm2', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'post_layernorm', 'norm', 'norm2', 'q_norm', 'cross_attn_input_layernorm', 'layer_norm2', 'cross_attn_post_attention_layernorm', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']
moazeldegwy/Qwen3-8B-LABD does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 2. Code Execution Engine

In [5]:
def execute_code_safely(code_str, test_cases=None, timeout_seconds=10):
    """Execute code in a completely isolated subprocess with hard OS timeout."""
    if test_cases is None:
        test_cases = []
    
    full_script = code_str + "\n\n" + "\n".join(test_cases)
    tmp_path = None
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_script)
            tmp_path = f.name
        
        result = subprocess.run(
            ["python", tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
        )
        
        if result.returncode == 0:
            return True, f"Execution Successful. Stdout:\n{result.stdout}", "None"
        else:
            return False, f"Error:\n{result.stderr[-500:]}", "Runtime"
    
    except subprocess.TimeoutExpired as e:
        # Kill any lingering child processes
        import psutil
        for proc in psutil.process_iter(['pid', 'cmdline']):
            try:
                if tmp_path and proc.info['cmdline'] and tmp_path in ' '.join(proc.info['cmdline']):
                    proc.kill()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        return False, "Execution Failed: Time Limit Exceeded", "Runtime"
    except Exception as e:
        return False, f"Execution Failed: {str(e)}", "Runtime"
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)

## 3. Load GRPO Dataset from Google Drive

In [6]:
from datasets import load_from_disk

drive_save_dir = "/root/grpo_train_dataset"

train_dataset = load_from_disk(drive_save_dir)
print(f"Loaded GRPO dataset: {train_dataset.column_names}, rows: {len(train_dataset)}")

Loaded GRPO dataset: ['prompt', 'answer'], rows: 476


In [7]:
train_dataset

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 476
})

In [8]:
train_dataset[84]

{'prompt': '<|im_start|>system\nYou are an autonomous Self-Correcting Python Agent. Your objective is to produce robust, correct Python code by iteratively generating, testing, and debugging solutions.\n\n### CORE PROTOCOL\nYou operate in a strictly defined loop: **THOUGHT** -> **ACTION** -> **OBSERVATION**.\n\n1. **THOUGHT (`<think>`):**\n   - **Initial Phase:** Analyze the problem requirements, identify edge cases, and formulate a plan.\n   - **Debugging Phase:** If previous execution failed, perform a **Forensic Analysis**. You MUST explicitly compare the "Expected" vs "Actual" outputs, identify the **Root Cause** of the logic error, and define a **Fix Strategy**.\n   - **Success Phase:** If the previous execution was successful, verify the solution is complete.\n\n2. **ACTION (`<execute>` or Code Block):**\n   - **Testing/Debugging:** If the solution is unverified or failed tests, output code inside `<execute>` tags.\n     - **CRITICAL:** You MUST append the user-provided `assert` 

In [9]:
train_dataset[85]

{'prompt': '<|im_start|>system\nYou are an autonomous Self-Correcting Python Agent. Your objective is to produce robust, correct Python code by iteratively generating, testing, and debugging solutions.\n\n### CORE PROTOCOL\nYou operate in a strictly defined loop: **THOUGHT** -> **ACTION** -> **OBSERVATION**.\n\n1. **THOUGHT (`<think>`):**\n   - **Initial Phase:** Analyze the problem requirements, identify edge cases, and formulate a plan.\n   - **Debugging Phase:** If previous execution failed, perform a **Forensic Analysis**. You MUST explicitly compare the "Expected" vs "Actual" outputs, identify the **Root Cause** of the logic error, and define a **Fix Strategy**.\n   - **Success Phase:** If the previous execution was successful, verify the solution is complete.\n\n2. **ACTION (`<execute>` or Code Block):**\n   - **Testing/Debugging:** If the solution is unverified or failed tests, output code inside `<execute>` tags.\n     - **CRITICAL:** You MUST append the user-provided `assert` 

In [10]:
# 1. Map a function to calculate the token length of each prompt
def calculate_length(example):
    # Tokenize the prompt to get the exact token count
    # add_special_tokens=False is used because your prompts already have <|im_start|> tags
    tokens = tokenizer(example['prompt'], truncation=False, add_special_tokens=False)
    return {'prompt_length': len(tokens['input_ids'])}

# Apply this to your dataset
dataset_with_lengths = train_dataset.map(calculate_length)

# 2. Calculate the statistics
lengths = dataset_with_lengths['prompt_length']
max_len = max(lengths)

threshold = 4096
exceeding_count = sum(1 for length in lengths if length > threshold)
percentage_exceeding = (exceeding_count / len(lengths)) * 100

print(f"Total examples: {len(lengths)}")
print(f"Maximum prompt length: {max_len} tokens")
print(f"Prompts > {threshold} tokens: {exceeding_count} ({percentage_exceeding:.2f}%)")

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

Total examples: 476
Maximum prompt length: 6877 tokens
Prompts > 4096 tokens: 20 (4.20%)


In [11]:
# Filter dataset
def calculate_length(example):
    tokens = tokenizer(example['prompt'], truncation=False, add_special_tokens=False)
    return {'prompt_length': len(tokens['input_ids'])}

dataset_with_lengths = train_dataset.map(calculate_length)
filtered_dataset = dataset_with_lengths.filter(lambda x: x['prompt_length'] <= 3072)
filtered_dataset = filtered_dataset.remove_columns(['prompt_length'])
print(f"Kept {len(filtered_dataset)} of {len(train_dataset)} examples")

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

Filter:   0%|          | 0/476 [00:00<?, ? examples/s]

Kept 388 of 476 examples


## 4. GRPO Reward Functions

In [12]:
def extract_code_xml(text):
    pattern = r"<execute>(.*?)</execute>"
    match = re.search(pattern, text, re.DOTALL)
    if match: return match.group(1).strip()
    pattern_md = r"```python(.*?)```"
    match_md = re.search(pattern_md, text, re.DOTALL)
    if match_md: return match_md.group(1).strip()
    return None

def correctness_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    
    for completion, test_cases in zip(completions, answer):
        code = extract_code_xml(completion)
        
        if not code:
            rewards.append(-1.0)
            continue
        
        try:
            success, _, _ = execute_code_safely(code, test_cases, timeout_seconds=10)
            rewards.append(3.0 if success else 0.0)
        except Exception:
            rewards.append(0.0)
    
    return rewards

def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        score = 0.0
        # If it tries to fake the system feedback, hit it with a heavy penalty
        if "<feedback>" in completion: 
            score -= 2.0
        rewards.append(score)
    return rewards

## 5. GRPO Training

In [22]:
# Free VRAM before training
import gc
gc.collect()
torch.cuda.empty_cache()

In [23]:
from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    temperature = 0.7,
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token, "<|im_end|>"],
    include_stop_str_in_output = True,
)

training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    dataloader_num_workers = 0,
    temperature = 0.7,
    learning_rate = 2e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    num_generations = 8,
    max_prompt_length = 3072,
    max_completion_length = 3072,
    num_train_epochs = 1,
    save_steps = 10,
    max_grad_norm = 1.0,
    report_to = "none",
    output_dir = "/mnt/training/grpo_outputs_8B",
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        correctness_reward_func,
        format_reward_func
    ],
    args = training_args,
    train_dataset = filtered_dataset,
)

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:
trainer.train()

## 6. Save & Push to Hub

In [26]:
model.push_to_hub("moazeldegwy/Qwen3-8B-LABD-GRPO-Adapters", token=hf_token)
tokenizer.push_to_hub("moazeldegwy/Qwen3-8B-LABD-GRPO-Adapters", token=hf_token)

In [27]:
# Merge to 16bit
model.push_to_hub_merged(
    "moazeldegwy/Qwen3-8B-LABD-GRPO",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token
)